In [11]:
from Chatbot import (
    retrieve_context,
    save_chat_history,
    fetch_chat_history,
    get_gemini_response,
)

In [12]:
import os
import numpy as np
import pandas as pd
import json
import time
import re
import random
from datetime import datetime
from typing import List, Dict, Any, Generator
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pinecone import Pinecone, ServerlessSpec
import fitz  # PyMuPDF
import docx2txt  # Word extraction
import serpapi
import langgraph
import langchain
import serpapi
import wikipedia
from groq import Groq
from langchain_groq import ChatGroq
from tavily import TavilyClient
from langchain_google_genai import ChatGoogleGenerativeAI
from google import genai

In [13]:
# ---------------- LOAD ENV VARIABLES ----------------
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("INDEX_NAME", "studybuddy")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
PINECONE_ENVIRONMENT = os.getenv("PINECONE_ENVIRONMENT", "us-east-1")
DEFAULT_TIMEZONE = os.getenv("DEFAULT_TIMEZONE", "UTC")

if not GEMINI_API_KEY:
    raise ValueError("❌ GEMINI_API_KEY not found!")
if not PINECONE_API_KEY:
    raise ValueError("❌ PINECONE_API_KEY not found!")

In [14]:
eval_questions = []
with open('generated_questions.txt', 'r', encoding='utf-8') as file:
    for line in file:
        # Remove newline character and convert to integer
        item = line.strip()
        eval_questions.append(item)

In [15]:
# Get answers for all questions
answers = []
for question in eval_questions:
    try:
        # Get response (handles both streaming and non-streaming)
        response = get_gemini_response(question, user_id="eval_user")
        
        # If it's a generator (streaming), collect all chunks
        if hasattr(response, '__iter__') and not isinstance(response, str):
            full_response = ""
            for chunk in response:
                if chunk:
                    full_response += chunk
            answers.append(full_response)
        else:
            answers.append(str(response))
            
        print(f"✅ Processed: {question[:50]}...")
        
    except Exception as e:
        print(f"❌ Error: {question[:50]}... - {e}")
        answers.append(f"ERROR: {str(e)}")

# Print results
print(f"\n✅ Completed {len(answers)}/{len(eval_questions)} questions")

✅ Processed: What is an alphabet in the context of automata the...
✅ Processed: What is the symbol used to denote the empty string...
✅ Processed: What is the Kleene star operator (Σ*) used for?...
✅ Processed: What is a formal language?...

🔧 Using tools:
   - search_wikipedia: empty language and language containing only the empty string
✅ Processed: According to the text, what is the difference betw...

🔧 Using tools:
   - web_search: software applications using automata theory finite automata
✅ Processed: List at least three different software application...
✅ Processed: Explain the difference between concatenating two s...
✅ Processed: Why is the empty string (ε) considered the identit...
✅ Processed: A language L is defined as {0ⁿ1ⁿ | n ≥ 1}. Write o...
[WARNING] Agent streaming failed: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed'

In [6]:
# pip install jsonschema torch torchvision jsonschema torchaudio --index-url https://download.pytorch.org/whl/cpu 

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# patch_torch.py
import sys
import torch
import os
from trulens.core import TruSession, Metric
from trulens.apps.app import TruApp

# ============================================
# 1. PATCH TORCH BEFORE ANY TRULENS IMPORTS
# ============================================

# Fix: Add Tensor attribute if missing
if not hasattr(torch, 'Tensor'):
    class Tensor:
        pass
    torch.Tensor = Tensor
    print("✅ Patched torch.Tensor")

# Also patch other common attributes that might be missing
if not hasattr(torch, 'FloatTensor'):
    torch.FloatTensor = torch.Tensor
if not hasattr(torch, 'LongTensor'):
    torch.LongTensor = torch.Tensor
if not hasattr(torch, 'DoubleTensor'):
    torch.DoubleTensor = torch.Tensor

# Patch device handling
if not hasattr(torch, 'device'):
    class Device:
        def __init__(self, device_str):
            self.type = device_str
        def __str__(self):
            return self.type
    torch.device = Device

# Patch dtype
if not hasattr(torch, 'dtype'):
    class Dtype:
        pass
    torch.dtype = Dtype
    torch.float32 = Dtype()
    torch.float64 = Dtype()

print(f"✅ PyTorch version: {torch.__version__}")
print("✅ torch.Tensor exists:", hasattr(torch, 'Tensor'))

# ============================================
# 2. NOW IMPORT TRULENS (without LangchainProvider)
# ============================================

# Import TruLens - use only what's available
from trulens.core import TruSession, Metric, Feedback

print("✅ TruLens core imports successful")

# ============================================
# 3. IMPORT OTHER DEPENDENCIES
# ============================================

from langchain_groq import ChatGroq
from Chatbot import retrieve_context

print("✅ All imports successful!")

# ============================================
# 4. SET UP GROQ PROVIDER MANUALLY
# ============================================

class GroqProvider:
    """Custom provider for Groq to use with TruLens"""
    
    def __init__(self, api_key):
        self.api_key = api_key
        self.client = None
        self._initialize_client()
    
    def _initialize_client(self):
        """Initialize Groq client"""
        try:
            from groq import Groq
            self.client = Groq(api_key=self.api_key)
            print("✅ Groq client initialized")
        except ImportError:
            print("⚠️ Groq not installed")
            self.client = None
    
    def generate(self, prompt: str, **kwargs):
        """Generate response using Groq"""
        if not self.client:
            raise ValueError("Groq client not initialized")
        
        response = self.client.chat.completions.create(
            model=kwargs.get('model', "meta-llama/llama-4-scout-17b-16e-instruct"),
            messages=[{"role": "user", "content": prompt}],
            temperature=kwargs.get('temperature', 0),
            **{k: v for k, v in kwargs.items() if k not in ['model', 'temperature']}
        )
        return response.choices[0].message.content

# ============================================
# 5. TEST THE SETUP
# ============================================

def test_setup():
    """Test if everything is working"""
    print("\n🔍 Testing setup...")
    
    # Test torch
    print(f"  torch.Tensor: {hasattr(torch, 'Tensor')}")
    print(f"  torch.FloatTensor: {hasattr(torch, 'FloatTensor')}")
    
    # Test TruLens
    try:
        session = TruSession()
        print("  ✅ TruSession created")
    except Exception as e:
        print(f"  ❌ TruSession failed: {e}")
    
    # Test custom provider
    try:
        import os
        api_key = os.getenv("GROQ_API_KEY")
        if api_key:
            provider = GroqProvider(api_key)
            print("  ✅ GroqProvider created")
        else:
            print("  ⚠️ GROQ_API_KEY not set")
    except Exception as e:
        print(f"  ❌ GroqProvider failed: {e}")

if __name__ == "__main__":
    test_setup()

ImportError: cannot import name 'TruSession' from 'trulens' (unknown location)

In [9]:
# trulens_working.py
import os
import numpy as np
import torch
from groq import Groq
from trulens.core import TruSession, Metric
from Chatbot import retrieve_context

# ============================================
# 1. PATCH TORCH (for Python 3.13)
# ============================================

if not hasattr(torch, 'Tensor'):
    class Tensor:
        pass
    torch.Tensor = Tensor
    print("✅ Patched torch.Tensor")

# ============================================
# 2. YOUR DATA
# ============================================

eval_questions = [
    "What is an alphabet in automata theory?",
    "What is the symbol used to denote the empty string?",
    "What is the Kleene star operator (Σ*) used for?",
    "What is a formal language?",
    "Difference between ∅ and {ε}?"
]

answers = [
    "In automata theory, an alphabet is a finite set of symbols...",
    "The symbol used to denote the empty string is ε (epsilon).",
    "The Kleene star operator represents the set of all strings...",
    "A formal language is a set of strings of symbols...",
    "The empty language (∅) has no strings, while {ε} has one string..."
]

# ============================================
# 3. SET API KEY
# ============================================


# ============================================
# 4. RETRIEVE CONTEXTS
# ============================================

print("🔄 Retrieving contexts...")
contexts = []
for q in eval_questions:
    context = retrieve_context(q, user_id="default_user_id", top_k=3)
    contexts.append([context])
print("✅ Contexts retrieved")

# ============================================
# 5. CREATE CUSTOM METRICS
# ============================================

client = Groq(api_key=GROQ_API_KEY)

def evaluate_context_relevance(input: str, context: list) -> float:
    """Evaluate if context is relevant to the question"""
    prompt = f"""Rate the relevance of the context to the question on a scale of 0-1.
    Return ONLY a number between 0 and 1.
    
    Question: {input}
    Context: {context[0][:500] if context else "No context"}
    
    Relevance score (0-1):"""
    
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    try:
        return float(response.choices[0].message.content.strip())
    except:
        return 0.5

def evaluate_answer_relevance(input: str, output: str) -> float:
    """Evaluate if the answer is relevant to the question"""
    prompt = f"""Rate the relevance of the answer to the question on a scale of 0-1.
    Return ONLY a number between 0 and 1.
    
    Question: {input}
    Answer: {output}
    
    Relevance score (0-1):"""
    
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    try:
        return float(response.choices[0].message.content.strip())
    except:
        return 0.5

def evaluate_groundedness(context: list, output: str) -> float:
    """Evaluate if the answer is grounded in the context"""
    prompt = f"""Rate how grounded the answer is in the context on a scale of 0-1.
    Return ONLY a number between 0 and 1.
    
    Context: {context[0][:500] if context else "No context"}
    Answer: {output}
    
    Groundedness score (0-1):"""
    
    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    try:
        return float(response.choices[0].message.content.strip())
    except:
        return 0.5

print("✅ Custom metrics created!")

# ============================================
# 6. SETUP TRULENS SESSION
# ============================================

session = TruSession()

# Create Metrics
f_context_relevance = (
    Metric(evaluate_context_relevance)
    .on_input()
    .on_context(collect_list=False)
    .aggregate(np.mean)
)

f_answer_relevance = (
    Metric(evaluate_answer_relevance)
    .on_input()
    .on_output()
)

f_groundedness = (
    Metric(evaluate_groundedness)
    .on_context(collect_list=False)
    .on_output()
)

print("✅ Metrics defined!")

# ============================================
# 7. RUN EVALUATION
# ============================================

print("\n🚀 Running Evaluation with Metrics...")

# Create records manually
records = []
for i, (q, a, c) in enumerate(zip(eval_questions, answers, contexts), 1):
    print(f"📝 Evaluating Q{i}: {q[:50]}...")
    
    # Calculate scores manually
    context_score = evaluate_context_relevance(q, c)
    answer_score = evaluate_answer_relevance(q, a)
    grounded_score = evaluate_groundedness(c, a)
    
    record = {
        "input": q,
        "output": a,
        "context": c,
        "context_relevance": context_score,
        "answer_relevance": answer_score,
        "groundedness": grounded_score
    }
    records.append(record)

# Convert to DataFrame
import pandas as pd
df = pd.DataFrame(records)

print("\n📊 Results:")
print(df[['input', 'context_relevance', 'answer_relevance', 'groundedness']].round(3))

# Averages
print("\n📈 Averages:")
print(f"Context Relevance: {df['context_relevance'].mean():.3f}")
print(f"Answer Relevance: {df['answer_relevance'].mean():.3f}")
print(f"Groundedness: {df['groundedness'].mean():.3f}")

# Save results
df.to_csv("trulens_results.csv", index=False)
print("\n💾 Results saved to trulens_results.csv")

# Try launching dashboard if available
try:
    print("\n🌐 Attempting to launch TruLens dashboard...")
    session.run_dashboard()
except Exception as e:
    print(f"Dashboard not available: {e}")
    print("Results are available in the DataFrame above.")

🔄 Retrieving contexts...
✅ Contexts retrieved


Metric implementation <function evaluate_context_relevance at 0x0000016D7A1525C0> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.
Metric implementation <function evaluate_answer_relevance at 0x0000016D0F2E62A0> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.
Metric implementation <function evaluate_groundedness at 0x0000016D0F2E5D00> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


✅ Custom metrics created!
✅ Metrics defined!

🚀 Running Evaluation with Metrics...
📝 Evaluating Q1: What is an alphabet in automata theory?...
📝 Evaluating Q2: What is the symbol used to denote the empty string...
📝 Evaluating Q3: What is the Kleene star operator (Σ*) used for?...
📝 Evaluating Q4: What is a formal language?...
📝 Evaluating Q5: Difference between ∅ and {ε}?...

📊 Results:
                                               input  context_relevance  \
0            What is an alphabet in automata theory?                0.8   
1  What is the symbol used to denote the empty st...                0.0   
2    What is the Kleene star operator (Σ*) used for?                0.9   
3                         What is a formal language?                0.0   
4                      Difference between ∅ and {ε}?                0.0   

   answer_relevance  groundedness  
0               0.9           1.0  
1               0.5           0.8  
2               0.8           0.8  
3             

C:\Users\Admin\AppData\Local\Temp\ipykernel_5808\3316379297.py:197: DeprecationWarning: Method `run_dashboard` has been renamed or moved to `trulens.dashboard.run.run_dashboard`.

  session.run_dashboard()


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://localhost:61544 .


In [ ]:
%pip install trulens trulens-apps-llamaindex trulens-providers-openai